# Proof of concept for detecting potential invaders in Belgium

In [1]:
# Run this once in your notebook cell
!pip install pygbif shap requests pandas matplotlib

In [28]:
# load in relevant libraries

import os
import sys
import pygbif
import requests
import bfm_model
import torch
import shap
import matplotlib.pyplot as plt
import pandas as pd

print("Libraries loaded successfully! :-)")

Libraries loaded successfully! :-)


In [24]:
import sys
import os
from pathlib import Path

# 1. Setup the absolute path to your repo root
repo_root = Path(r"C:\Users\simon\Documents\GitHub\horizon-scanner\bfm-model")
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

# 2. Corrected Import from model.py
try:
    # Based on your file list, the core logic is in model.py
    from bfm_model.bfm.model import BioAnalyst
    print("✅ Success! BioAnalyst class imported from model.py")
except ImportError:
    try:
        from bfm_model.bfm.model import BFM
        print("✅ Success! BFM class imported from model.py")
    except ImportError as e:
        print(f"❌ Still can't find the class. Let's inspect model.py directly:")
        import bfm_model.bfm.model as model_file
        print(f"Functions/Classes in model.py: {[name for name in dir(model_file) if not name.startswith('_')]}")

✅ Success! BFM class imported from model.py


In [30]:
import os
import glob
from pathlib import Path
from omegaconf import OmegaConf
from bfm_model.bfm.model import BFM

# 1. Search for the config file automatically
repo_root = Path(r"C:\Users\simon\Documents\GitHub\horizon-scanner\bfm-model")
# We look for any .yaml file that might be the biocube config
search_pattern = str(repo_root / "**" / "*.yaml")
possible_configs = glob.glob(search_pattern, recursive=True)

# Filter for the one likely to be the biocube variables
target_config = None
for p in possible_configs:
    if "biocube" in p.lower() or "variable" in p.lower():
        target_config = p
        break

if target_config:
    print(f"🎯 Found config at: {target_config}")
    cfg = OmegaConf.load(target_config)
    
    # 2. Map the config to the BFM constructor
    # We use the specific keys identified in your previous TypeError
    try:
        model = BFM(
            surface_vars=list(cfg.surface.vars),
            edaphic_vars=list(cfg.edaphic.vars),
            atmos_vars=list(cfg.atmos.vars),
            climate_vars=list(cfg.climate.vars),
            species_vars=list(cfg.species.vars),
            vegetation_vars=list(cfg.vegetation.vars),
            land_vars=list(cfg.land.vars),
            agriculture_vars=list(cfg.agriculture.vars),
            forest_vars=list(cfg.forest.vars),
            redlist_vars=list(cfg.redlist.vars),
            misc_vars=list(cfg.misc.vars),
            atmos_levels=cfg.atmos.levels,
            species_num=cfg.species.num
        )
        print("✅ BFM initialized successfully!")
    except Exception as e:
        print(f"❌ Initialization failed: {e}")
        print("\nHint: Your config might use different keys. Here are the top-level keys found in the YAML:")
        print(cfg.keys())
else:
    print("❌ Could not find a BioCube YAML file. Please check if 'bfm_model/bfm/configs' has any .yaml files.")

❌ Could not find a BioCube YAML file. Please check if 'bfm_model/bfm/configs' has any .yaml files.


In [ ]:
# Initial watchlist
watchlist = ["Vespa orientalis", "Solenopsis invicta", "Vespa velutina"]
target = "BE"

def check_belgium(species_list):
    new_threats = []
    for sp in species_list:
        # We search specifically for established occurrences in Belgium
        res = pygbif.occurrences.search(scientificName=sp, country=target, limit=1)
        if res['count'] == 0:
            print(f"Found potential invader: {sp}")
            new_threats.append(sp)
        else:
            print(f"Already in Belgium: {sp}")
    return new_threats

active_threats = check_belgium(watchlist)

Already in Belgium: Vespa orientalis
Found potential invader: Solenopsis invicta
Already in Belgium: Vespa velutina


In [9]:
def get_interactions_comprehensive(species_name):
    """
    Queries GloBI using both source and target parameters 
    to ensure we capture the full ecological niche.
    """
    if isinstance(species_name, list):
        species_name = species_name
        
    # We will try both sourceTaxon and targetTaxon to be thorough
    interactions = set()
    endpoints = ["sourceTaxon", "targetTaxon"]
    
    print(f"Searching GloBI for: {species_name}...")
    
    for param in endpoints:
        url = "https://api.globalbioticinteractions.org/interaction"
        # Removed the limit to get all available records
        query_params = {
            param: species_name, 
            "type": "json.v2"
        }
        
        try:
            r = requests.get(url, params=query_params, timeout=20)
            if r.status_code == 200:
                raw_data = r.json()
                # Check if 'data' exists and has content
                if "data" in raw_data and len(raw_data["data"]) > 0:
                    # Index 2 is the interaction type (e.g., 'preysOn')
                    for row in raw_data["data"]:
                        interactions.add(row)
            else:
                print(f"  - API returned {r.status_code} for {param}")
        except Exception as e:
            print(f"  - Request failed for {param}: {e}")
            
    result_list = list(interactions)
    if not result_list:
        print(f"⚠️ No interactions found in GloBI for '{species_name}'.")
        # Fallback: Sometimes GloBI prefers Genus level for general traits
        print(f"Hint: If this is a rare species, the BFM will rely on Genus-level traits.")
        
    return result_list

# Test again with the first threat
if active_threats:
    species_to_test = active_threats # e.g., "Vespa orientalis"
    all_interactions = get_interactions_comprehensive(species_to_test)
    print(f"\nFinal unique interaction types found: {all_interactions}")

Searching GloBI for: ['Solenopsis invicta']...
⚠️ No interactions found in GloBI for '['Solenopsis invicta']'.
Hint: If this is a rare species, the BFM will rely on Genus-level traits.

Final unique interaction types found: []


In [ ]:
# Load the model weights (this might take a moment to load into memory)
# If this is the first time, you may need to run:
# !python -m bfm.download_assets --model ias-v2
model = bfm_model.("bfm-v2-ias")

# Predict for our filtered list
results = []
for species in active_threats:
    pred = model.predict_invasion_risk(
        species=species,
        region="BE",
        biotic_features=get_interactions(species)
    )
    results.append(pred)
    print(f"Risk Score for {species}: {pred.score:.2f}")

AttributeError: module 'bfm_model' has no attribute 'bfm'